In [2]:
SEASON = "2025-26"

WEST = {
    "DAL", "DEN", "GSW", "HOU", "LAC", "LAL", "MEM", "MIN",
    "NOP", "OKC", "PHX", "POR", "SAC", "SAS", "UTA",
}

EAST = {
    "ATL", "BOS", "BKN", "CHA", "CHI", "CLE", "DET", "IND",
    "MIA", "MIL", "NYK", "ORL", "PHI", "TOR", "WAS",
}

from nba_api.stats.static import teams
import pandas as pd
from functools import lru_cache
from nba_api.stats.endpoints import teaminfocommon


@lru_cache(maxsize=None)
def get_all_teams_info():
    return pd.DataFrame(teams.get_teams())


@lru_cache(maxsize=None)
def team_abbrev_by_id():
    return get_all_teams_info().set_index("id")["abbreviation"]


@lru_cache(maxsize=None)
def get_team_info(team_id, season):
    return teaminfocommon.TeamInfoCommon(
        team_id=team_id,
        season_nullable=season,
        season_type_nullable="Regular Season",
    ).team_info_common.get_data_frame()


def team_id(abbrev: str) -> int:
    return teams.find_team_by_abbreviation(abbrev)["id"]

get_all_teams_info().head()

,id,full_name,abbreviation,nickname,city,state,year_founded
0,1610612737,Atlanta Hawks,ATL,Hawks,Atlanta,Georgia,1949
1,1610612738,Boston Celtics,BOS,Celtics,Boston,Massachusetts,1946
2,1610612739,Cleveland Cavaliers,CLE,Cavaliers,Cleveland,Ohio,1970
3,1610612740,New Orleans Pelicans,NOP,Pelicans,New Orleans,Louisiana,2002
4,1610612741,Chicago Bulls,CHI,Bulls,Chicago,Illinois,1966


# regular season records

In [3]:
from nba_api.stats.endpoints import leaguedashteamstats

@lru_cache(maxsize=None)
def get_all_records(season):
    records = leaguedashteamstats.LeagueDashTeamStats(
        season=season,
        season_type_all_star="Regular Season",
        measure_type_detailed_defense="Base",
        per_mode_detailed="Totals",
    ).get_data_frames()[0]

    #records = records[
    #    ["TEAM_ID", "TEAM_NAME", "GP", "W", "L", "W_PCT", "PLUS_MINUS"]
    #].sort_values("W_PCT", ascending=False)

    return records.sort_values("W_PCT", ascending=False)


def add_conference(records):
    records = records.copy()

    conference_by_abbrev = {abbrev: "West" for abbrev in WEST}
    conference_by_abbrev.update({abbrev: "East" for abbrev in EAST})
    records["TEAM_CONFERENCE"] = records["TEAM_ID"].map(team_abbrev_by_id()).map(conference_by_abbrev)

    return records

add_conference(get_all_records(SEASON)).head()

,TEAM_ID,TEAM_NAME,GP,W,L,W_PCT,MIN,FGM,FGA,FG_PCT,...,AST_RANK,TOV_RANK,STL_RANK,BLK_RANK,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,TEAM_CONFERENCE
20,1610612760,Oklahoma City Thunder,82,64,18,0.780,3971.0,3534,7307,0.484,...,19,2,2,6,8,7,9,5,1,West
26,1610612759,San Antonio Spurs,82,62,20,0.756,3946.0,3562,7371,0.483,...,9,4,25,8,11,4,12,3,2,West
8,1610612765,Detroit Pistons,82,60,22,0.732,3961.0,3560,7340,0.485,...,11,21,1,1,29,30,4,8,3,East
1,1610612738,Boston Celtics,82,56,26,0.683,3946.0,3456,7398,0.467,...,27,1,28,11,3,9,27,19,4,East
7,1610612743,Denver Nuggets,82,54,28,0.659,3986.0,3570,7196,0.496,...,4,3,30,28,13,11,3,1,7,West


# Advanced stats

In [4]:
advanced = leaguedashteamstats.LeagueDashTeamStats(
    season=SEASON,
    season_type_all_star="Regular Season",
    measure_type_detailed_defense="Advanced",
    per_mode_detailed="Per100Possessions",
).get_data_frames()[0]

# available advanced statss
print(advanced.columns.tolist())

net_ratings = advanced[
    ["TEAM_ID", "TEAM_NAME", "GP", "W", "L", "NET_RATING"]
].sort_values("NET_RATING", ascending=False)

print(net_ratings.head())

['TEAM_ID', 'TEAM_NAME', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'E_OFF_RATING', 'OFF_RATING', 'E_DEF_RATING', 'DEF_RATING', 'E_NET_RATING', 'NET_RATING', 'AST_PCT', 'AST_TO', 'AST_RATIO', 'OREB_PCT', 'DREB_PCT', 'REB_PCT', 'TM_TOV_PCT', 'EFG_PCT', 'TS_PCT', 'E_PACE', 'PACE', 'PACE_PER40', 'POSS', 'PIE', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'OFF_RATING_RANK', 'DEF_RATING_RANK', 'NET_RATING_RANK', 'AST_PCT_RANK', 'AST_TO_RANK', 'AST_RATIO_RANK', 'OREB_PCT_RANK', 'DREB_PCT_RANK', 'REB_PCT_RANK', 'TM_TOV_PCT_RANK', 'EFG_PCT_RANK', 'TS_PCT_RANK', 'PACE_RANK', 'PIE_RANK']
       TEAM_ID              TEAM_NAME  GP   W   L  NET_RATING
20  1610612760  Oklahoma City Thunder  82  64  18        11.1
26  1610612759      San Antonio Spurs  82  62  20         8.4
8   1610612765        Detroit Pistons  82  60  22         8.4
1   1610612738         Boston Celtics  82  56  26         8.3
19  1610612752        New York Knicks  82  53  29         6.4


# regular season head-to-head matchup

In [5]:
from nba_api.stats.endpoints import leaguegamefinder


def h2h_record(id_1, id_2):    
    matchups = leaguegamefinder.LeagueGameFinder(
        player_or_team_abbreviation="T",
        season_nullable=SEASON,
        season_type_nullable="Regular Season",
        team_id_nullable=id_1,
        vs_team_id_nullable=id_2,
    ).get_data_frames()[0]

    record = matchups["WL"].value_counts()

    print("wins:", record.get("W", 0))
    print("losses:", record.get("L", 0))
    return matchups[["GAME_DATE", "MATCHUP", "WL", "PTS", "PLUS_MINUS"]]

h2h_record(team_id("OKC"), team_id("SAS"))

wins: 1
losses: 4


,GAME_DATE,MATCHUP,WL,PTS,PLUS_MINUS
0,2026-02-04,OKC @ SAS,L,106,-10.0
1,2026-01-13,OKC vs. SAS,W,119,21.0
2,2025-12-25,OKC vs. SAS,L,102,-15.0
3,2025-12-23,OKC @ SAS,L,110,-20.0
4,2025-12-13,SAS @ OKC,L,109,-2.0


# Predictor

In [6]:
def net_rating(team):
    return float(net_ratings.loc[net_ratings["TEAM_ID"].eq(team), "NET_RATING"].iloc[0])


def predict(id_1, id_2, season):
    return id_1 if net_rating(id_1) > net_rating(id_2) else id_2


def winner_loser(team_1, team_2, season):
    winner = predict(team_1, team_2, season)
    return winner, team_1 if winner == team_2 else team_2


def full_playoffs_predict(season):
    records = add_conference(get_all_records(season))
    team_info = get_all_teams_info()[[
        "id", "full_name", "abbreviation", "nickname"
    ]].rename(columns={"id": "TEAM_ID"})
    records = records.merge(team_info, on="TEAM_ID", how="left")

    team_names = records.set_index("TEAM_ID")["TEAM_NAME"].to_dict()
    team_abbrevs = records.set_index("TEAM_ID")["abbreviation"].to_dict()

    def seed_play_in(conference):
        top10 = records[records["TEAM_CONFERENCE"].eq(conference)].head(10).copy().reset_index(drop=True)
        seeds = [int(team) for team in top10["TEAM_ID"]]

        seed_7, loser_78 = winner_loser(seeds[6], seeds[7], season)
        winner_910, _ = winner_loser(seeds[8], seeds[9], season)
        seed_8, _ = winner_loser(loser_78, winner_910, season)

        playoff_ids = seeds[:6] + [seed_7, seed_8]
        seed_by_team = {team: seed for seed, team in enumerate(playoff_ids, start=1)}
        play_in = [
            ("7/8", seeds[6], seeds[7], seed_7),
            ("9/10", seeds[8], seeds[9], winner_910),
            ("8 seed", loser_78, winner_910, seed_8),
        ]
        return playoff_ids, seed_by_team, play_in

    west_playoffs, west_seeds, west_play_in = seed_play_in("West")
    east_playoffs, east_seeds, east_play_in = seed_play_in("East")
    seeds = {**west_seeds, **east_seeds}

    def label(team):
        return f"({seeds[team]}) {team_abbrevs[team]}"

    def game(team_1, team_2):
        winner, loser = winner_loser(team_1, team_2, season)
        return {"winner": winner, "loser": loser, "team_1": team_1, "team_2": team_2}

    def play_conference(playoff_ids):
        first_round = [
            game(playoff_ids[0], playoff_ids[7]),
            game(playoff_ids[3], playoff_ids[4]),
            game(playoff_ids[2], playoff_ids[5]),
            game(playoff_ids[1], playoff_ids[6]),
        ]
        semifinals = [
            game(first_round[0]["winner"], first_round[1]["winner"]),
            game(first_round[2]["winner"], first_round[3]["winner"]),
        ]
        finals = [game(semifinals[0]["winner"], semifinals[1]["winner"])]
        return first_round, semifinals, finals

    west_rounds = play_conference(west_playoffs)
    east_rounds = play_conference(east_playoffs)
    nba_finals = game(west_rounds[2][0]["winner"], east_rounds[2][0]["winner"])

    def print_game(result):
        winner = result["winner"]
        loser = result["loser"]
        print(f"  {label(winner)} def. {label(loser)}")

    def print_play_in(conference, play_in):
        print(f"{conference} Play-In")
        for name, team_1, team_2, winner in play_in:
            loser = team_1 if winner == team_2 else team_2
            print(f"  {name}: {team_abbrevs[winner]} def. {team_abbrevs[loser]}")

    def print_round(title, west_games, east_games=None):
        print(f"\n{title}")
        if east_games is not None:
            print("West")
        for result in west_games:
            print_game(result)
        if east_games is not None:
            print("East")
            for result in east_games:
                print_game(result)

    print_play_in("West", west_play_in)
    print_play_in("East", east_play_in)
    print_round("First Round", west_rounds[0], east_rounds[0])
    print_round("Conference Semifinals", west_rounds[1], east_rounds[1])
    print_round("Conference Finals", west_rounds[2], east_rounds[2])
    print_round("NBA Finals", [nba_finals])

    champion = nba_finals["winner"]
    print(f"\nChampion: {team_names[champion]} ({team_abbrevs[champion]})")
    return team_names[champion]


full_playoffs_predict(SEASON)

West Play-In
  7/8: PHX def. LAC
  9/10: POR def. GSW
  8 seed: LAC def. POR
East Play-In
  7/8: ORL def. PHI
  9/10: CHA def. MIA
  8 seed: CHA def. PHI

First Round
West
  (1) OKC def. (8) LAC
  (5) HOU def. (4) LAL
  (3) DEN def. (6) MIN
  (2) SAS def. (7) PHX
East
  (1) DET def. (8) CHA
  (4) CLE def. (5) ATL
  (3) NYK def. (6) TOR
  (2) BOS def. (7) ORL

Conference Semifinals
West
  (1) OKC def. (5) HOU
  (2) SAS def. (3) DEN
East
  (1) DET def. (4) CLE
  (2) BOS def. (3) NYK

Conference Finals
West
  (1) OKC def. (2) SAS
East
  (1) DET def. (2) BOS

NBA Finals
  (1) OKC def. (1) DET

Champion: Oklahoma City Thunder (OKC)


'Oklahoma City Thunder'

In [7]:
records = leaguedashteamstats.LeagueDashTeamStats(
        season=SEASON,
        season_type_all_star="Regular Season",
        measure_type_detailed_defense="Base",
        per_mode_detailed="Totals",
    ).get_data_frames()[0]

records

,TEAM_ID,TEAM_NAME,GP,W,L,W_PCT,MIN,FGM,FGA,FG_PCT,...,REB_RANK,AST_RANK,TOV_RANK,STL_RANK,BLK_RANK,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK
0,1610612737,Atlanta Hawks,82,46,36,0.561,3951.000000,3575,7541,0.474,...,18,1,10,5,18,17,14,26,6,12
1,1610612738,Boston Celtics,82,56,26,0.683,3946.000000,3456,7398,0.467,...,3,27,1,28,11,3,9,27,19,4
2,1610612751,Brooklyn Nets,82,20,62,0.244,3951.000000,3071,6933,0.443,...,30,26,29,22,23,23,26,21,30,28
3,1610612766,Charlotte Hornets,82,44,38,0.537,3951.000000,3357,7292,0.460,...,5,15,25,29,21,9,10,18,13,8
4,1610612741,Chicago Bulls,82,31,51,0.378,3951.000000,3476,7417,0.469,...,9,7,23,23,11,26,8,24,12,22
5,1610612739,Cleveland Cavaliers,82,52,30,0.634,3951.000000,3554,7381,0.482,...,11,8,9,16,13,6,12,7,4,9
6,1610612742,Dallas Mavericks,82,26,56,0.317,3971.000000,3432,7366,0.466,...,10,22,18,24,10,21,2,5,23,23
7,1610612743,Denver Nuggets,82,54,28,0.659,3986.000000,3570,7196,0.496,...,14,4,3,30,28,13,11,3,1,7
8,1610612765,Detroit Pistons,82,60,22,0.732,3961.000000,3560,7340,0.485,...,8,11,21,1,1,29,30,4,8,3
9,1610612744,Golden State Warriors,82,37,45,0.451,3961.000000,3357,7280,0.461,...,21,6,27,2,26,5,13,25,22,20


In [ ]:
from nba_api.stats.endpoints import leaguegamelog
from nba_api.stats.endpoints import commonplayoffseries

def eval_playoffs(season):
    playoff_series = commonplayoffseries.CommonPlayoffSeries(season=season)
    df_series = playoff_series.get_data_frames()[0]
    
    return df_series


eval_playoffs(SEASON)

,GAME_ID,HOME_TEAM_ID,VISITOR_TEAM_ID,SERIES_ID,GAME_NUM
0,0042500101,1610612765,1610612753,004250010,1
1,0042500102,1610612765,1610612753,004250010,2
2,0042500103,1610612753,1610612765,004250010,3
3,0042500104,1610612753,1610612765,004250010,4
4,0042500105,1610612765,1610612753,004250010,5
...,...,...,...,...,...
80,0042500401,1610612759,1610612752,004250040,1
81,0042500402,1610612759,1610612752,004250040,2
82,0042500403,1610612752,1610612759,004250040,3
83,0042500404,1610612752,1610612759,004250040,4
